In [ ]:
import os, sys
from pathlib import Path
from importlib.metadata import version
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import gc

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import rcParams

rcParams['pdf.fonttype'] = 42          # editable text in PDFs
%config InlineBackend.print_figure_kwargs = {'bbox_inches': 'tight', 'dpi': 90}   # inline display only; files are saved at 300 dpi

import plotly.io as pio
pio.renderers.default = "notebook_connected"

ROOT0 = Path('/home/flavio/uv/cell_cell_communication')
ROOT_SRC = ROOT0 / 'src'
if str(ROOT_SRC) not in sys.path:
    sys.path.append(str(ROOT_SRC))

from libs.single_cell_heart_dvlp_lib import SingleCellHeart, HEART_PARAMS, HEART_LINEAGE, HEART_GENES, HEART_FILES
from libs.single_cell_lib import cmap_byr, cmap_lbwr
from libs.matplotlib_figure_style import apply_figure_style
from libs.Basic import create_dir, pdreadcsv

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.width', 250)
pd.set_option('display.max_rows', 200)
print({p: version(p) for p in ['scanpy', 'anndata', 'plotly']})

In [ ]:
force=False
verbose=False

apply_figure_style()
sch = SingleCellHeart(ROOT0, analysis='heart03')

In [ ]:
df_regions = sch.read_tsv('region_labels', verbose=verbose)

print(df_regions.shape)
df_regions.head(3)

In [ ]:
source = df_regions['abundance_source'].iloc[0]
adata_vis = sch.read_h5ad_slots('c2l_map', skip=('layers',), verbose=verbose) if source == 'c2l' else \
            sch.read_h5ad('visium', set_self=False, verbose=verbose)
adata_vis = sch.remove_mt_genes(adata_vis)
names_auto = sch.add_region_labels_from_table(df_regions, adata_vis)

In [ ]:
df_abund = sch.get_abundance(adata_vis, source=source)
df_region_info = sch.read_region_info(source=source)
print(df_region_info.shape)
df_region_info

In [ ]:
fname = 'heart_lineage.tsv'

root_table = ROOT0 / 'tables'
dfhl = pdreadcsv(fname, root_table, verbose=True)
dfhl

In [ ]:
names = dict(zip(df_region_info.region_id, df_region_info.region_name)) if df_region_info is not None else names_auto

adata_expr = sch.prepare_expression(adata_vis)
sections_rep = sch.representative_sections()
sections_show = [s for s in sections_rep if sch.section_title(s, short=True).split(' | ')[0] in ('w6', 'w8', 'w10', 'w12')]
print(adata_expr)
print("abundance source:", source, "| sections shown in the per-region figures:", sections_show)
adata_expr.obs['region_label'].value_counts().sort_index().to_frame('n_spots').T

### Rank region - Region Signatures

In [ ]:
force=False
verbose=False

# "rest" (default): all other cells/spots pooled ("one vs rest").
# each Rxx verssus all others Rx1, Rx2, Rxn
# "wilcoxon" test: rank-based (Mann–Whitney U); robust to outliers and the usual choice for single-cell data.
df_rank_region, region_sig = sch.calc_df_region_signature(adata_expr, force=force, verbose=verbose)

# print(np.array([x for x in region_sig['R00']]))

print(df_rank_region.shape)
df_rank_region.head(10)

### Region Table

In [ ]:
df_region_table = sch.calc_df_region_table(adata_expr, df_rank_region, region_sig, df_abund, df_region_info,
                                           force=force, verbose=verbose)
# print(np.array(df_region_table.columns))
cols = ['region_id', 'region_name', 'compartment', 'n_spots', 'median_UMIs',  'median_cells_per_spot', 
        'n_signature_genes', 'signature_genes', 'enriched_cell_types', 'functional_description']
df_region_table[cols]

In [ ]:
df_depth = sch.calc_df_region_depth(adata_expr, df_abund)
df_depth.T

In [ ]:
sections_rep

In [ ]:
sch.params['library_main']

In [ ]:
force=False
verbose=True

lib = sch.params['library_main']

fig = sch.plotly_region_map(adata_expr, adata_vis, lib, df_region_table=df_region_table, fontsize_title=20, fontsize_legend=12,
                            fontsize_hover=13, reset_button=True, force=force, verbose=verbose)

if fig:
    fig.show()

### 3. Subclustering each region (own UMAP)

In [ ]:
force=False
verbose=False

subs, df_grid = sch.run_subregions(adata_expr, force=force, verbose=verbose)
cols = ['region_id', 'resolution', 'k', 'min_size']
df_grid[df_grid.chosen][cols].head(8)

In [ ]:
len(subs), subs['R00']

### 4. Subregions

#### Gene sets of each subregion (partition)

Per subregion (Wilcoxon, subregion vs rest of its region), universe = genes detected in >= 1% of the region's spots:

| set | definition |
|---|---|
| region_signature | in the region signature (section 2) |
| subregion_signature | logFC >= 0.75, FDR <= 0.05, detected in >= 20% of the subregion; not in the region signature |
| not_present | detected in >= 30% of the rest of the region, <= 5% of the subregion, FDR <= 0.05 |
| other | the rest of the universe |

In [ ]:
force=False
verbose=False

df_gene_sets = sch.calc_df_subregion_gene_sets(adata_expr, region_sig, force=force, verbose=verbose)
df_set_counts = df_gene_sets.groupby(['sub_region_id', 'gene_set'], sort=False).size().unstack(fill_value=0)
df_set_counts['total'] = df_set_counts.sum(1)

n_universe = df_gene_sets.groupby('sub_region_id', sort=False).size()
assert (df_set_counts['total'] == n_universe.loc[df_set_counts.index]).all()

print(f"{len(df_set_counts)} subregions; the 4 sets partition every universe")

df_set_counts.head(10)

In [ ]:
force=False
verbose=True

path_desc = sch.file_path('subregion_descriptions')
df_descriptions = sch.read_tsv('subregion_descriptions', verbose=verbose) if path_desc.exists() else None
df_sub_table = sch.calc_df_subregion_table(adata_expr, df_gene_sets, df_abund, df_descriptions=df_descriptions,
                                           force=force, verbose=verbose)
print(df_sub_table.status.str.split(';').str[0].value_counts().to_dict())
cols = ['sub_region_id', 'n_spots', 'coherence_ratio', 'depth_ratio', 'border_region', 'border_frac', 'border_rel',
        'n_subregion_signature', 'n_not_present', 'status']
df_sub_table[cols].head(12)

### 6. Each region: UMAP and map of its subregions

Per region: UMAP coloured by subregion, UMAP coloured by section, and the subregions on the H&E of four sections (w6, w8, w10, w12; a region spans up to 38 sections, so all of them in one figure would be unreadable). The UMAPs show every spot of the region. Click a legend entry to hide / isolate a subregion in all panels.

In [ ]:
force=False
verbose=False

adata_show = adata_vis[adata_vis.obs['section'].isin(sections_show)].copy()
adata_show.uns['spatial'] = {k: v for k, v in adata_vis.uns['spatial'].items() if k in sections_show}

cnt = 0
for region, adata_sub in subs.items():
    fig = sch.plotly_region_subregions(region, adata_sub, adata_show, 
                                       df_sub_table=df_sub_table, df_region_table=df_region_table,
                                       ncols=2, nrows=3,
                                       fontsize_title=20, fontsize_subplot_title=16, fontsize_axis=14, fontsize_legend=12,
                                       fontsize_colorbar=12, fontsize_hover=13, force=force, verbose=verbose)
    if fig: fig.show()

    cnt += 1
    if cnt == 4: break

### LRP1 - PLAT

LRP1 (low-density lipoprotein receptor-related protein 1) and PLAT (tissue-type plasminogen activator, or tPA) interact directly to regulate blood clotting, arterial wall integrity, and heart tissue repair after injury. [1](https://www.ahajournals.org/doi/10.1161/ATVBAHA.118.311882), [2](https://pmc.ncbi.nlm.nih.gov/articles/PMC6497734/)
Together, these proteins play a critical role in how arteries prevent blockages and how the heart recovers from reduced blood flow (ischemia). [1](https://pmc.ncbi.nlm.nih.gov/articles/PMC6058925/), [2](https://pmc.ncbi.nlm.nih.gov/articles/PMC6497734/)

1. How PLAT (tPA) and LRP1 Interact

- Clearance and Recycling. PLAT is an enzyme that breaks down blood clots (fibrinolysis). LRP1 acts as a cellular receptor that binds to PLAT—often when PLAT is paired with its inhibitor, PAI-1—and clears it from the bloodstream into cells.

- Cell Signaling. Beyond clearing proteins, PLAT binding to LRP1 triggers intracellular signaling cascades. This interaction influences cell survival, inflammation, and how cells remodel the surrounding tissue matrix. [1](https://pmc.ncbi.nlm.nih.gov/articles/PMC6497734/)

2. Role in the Arteries

- Plaque and Atherosclerosis. In the arterial wall, LRP1 is heavily expressed in vascular smooth muscle cells (VSMCs) and macrophages. It helps clear atherogenic lipoproteins and regulates growth factor receptors (like PDGFR-β and TGF-β) to prevent excessive smooth muscle cell migration and arterial plaque buildup. [1](https://www.frontiersin.org/journals/cardiovascular-medicine/articles/10.3389/fcvm.2021.682389/full), [2](https://www.sci-hub.red/10.1161/01.atv.0000154280.62072.fd), [3](https://insight.jci.org/articles/view/164751)

- Aneurysm Protection. LRP1 maintains the structural integrity of the arterial wall by controlling extracellular matrix turnover and smooth muscle contractility. Deficiencies or mutations in the LRP1 gene are linked to weakened arterial walls, aortic aneurysms, and arterial dissections. [1](https://journals.physiology.org/doi/abs/10.1152/ajpheart.00343.2026), [2](https://www.ahajournals.org/doi/10.1161/ATVBAHA.118.311882), [3](https://insight.jci.org/articles/view/164751)
- Clot Regulation. PLAT prevents dangerous arterial thrombosis by converting plasminogen into plasmin to dissolve fibrin clots, while endothelial LRP1 helps regulate local PLAT availability.

3. Role in the Heart

- Ischemia and Heart Attack Recovery. Following an acute myocardial infarction (heart attack), LRP1 expression increases in heart tissue. When activated by serine protease complexes (like PLAT-PAI-1) or therapeutic agonists, LRP1 triggers pro-survival pathways (such as Akt and ERK).

- Limiting Scar Damage. LRP1 signaling in the heart dampens excessive inflammation, promotes the clearance of dead cells, and guides fibroblasts to form a stable, protective scar rather than adverse heart remodeling that leads to heart failure. [1](https://www.sciencedirect.com/science/article/abs/pii/S0022356526011389), [2](https://pmc.ncbi.nlm.nih.gov/articles/PMC6497734/), [3](https://pmc.ncbi.nlm.nih.gov/articles/PMC6058925/)

<br>

![LIANA results](/home/flavio/uv/cell_cell_communication/heart_dvlp/figures/heart04_lr_top_local_scores_V11A20-401_A1.png)
